# 第 1 集 — 没有 `dac~`:声音就是一个 array

⏱ 约 25 分钟 · 难度 ★☆☆

**这一集控制的音乐元素:pitch(音高)、时长、音量。**

**Max 里的对应 patch:** `cycle~ 110` → `*~ 0.3` → `dac~`,开 2 秒再关。

**你会听到:** 一个干净、低沉的 "嗡——"(A2,110 Hz),持续 2 秒。之后你会自己做出三个音的小旋律和一个和弦。

**学完你能做到:**
- 说清楚 Max 的 signal 和 Python 的 array 有什么不同
- 从零写出一个 sine,并听到、看到它
- 改 pitch、时长、音量
- 把几个音接成旋律、叠成和弦
- 把声音存成 `.wav` 文件

---

### Max 和 Python 最大的区别

| | Max / RNBO | Python |
|---|---|---|
| 时间 | 自己在走,打开 `dac~` 就一直流 | 没有 "正在走的时间" |
| 声音 | 一个一个 sample 实时送出去 | **一次把整段算完**,放进一个 array |
| 听 | 立刻听到 | 算完以后再播放 |

所以在 Python 里做 synth,本质上是:**算出一长串数字,然后播放它。**

## 0. 怎么用这个 notebook

- 每个灰色格子叫一个 **cell**。点进去,按 **Shift + Enter** 运行它。
- **从上往下按顺序运行。** 后面的 cell 会用到前面 cell 里算出来的东西。
- 改了某个 cell 里的数字之后,要 **重新运行那个 cell 以及它下面用到它的 cell**,改动才会生效。这一点和 Max 不一样:Max 改了数字立刻生效,这里要 "重新算一遍"。
- ⚠️ 先把系统音量调小一点再开始。

下面这格是 "载入工具":

| 这一行 | 是什么 | 像 Max 里的 |
|---|---|---|
| `numpy`(简写 `np`) | 对一整串数字做数学运算 | 所有带 `~` 的数学 object |
| `matplotlib`(简写 `plt`) | 画图 | `scope~` |
| `Audio` | 播放器 | `dac~` |
| `wavfile` | 读写 wav 文件 | `sfrecord~` / `buffer~` 的 write |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.io import wavfile

## 1. 什么是 array

array 就是 **一串按顺序排好的数字**。先做一个只有 4 个数的。

要记住的四件事:

1. **整串一起算。** `a * 2` 会把里面每个数都乘 2。不需要循环。
2. **`len(a)`** 是它有多少个数。
3. **方括号取数,从 0 开始数。** `a[0]` 是第一个,`a[3]` 是第四个。
4. **`a[:2]`** 是 "从头取到第 2 个之前",也就是前两个。这叫 slicing。

运行下面这格,对照输出看。

In [ ]:
a = np.array([10, 20, 30, 40])

print("a        =", a)
print("a * 2    =", a * 2)
print("a + 1    =", a + 1)
print("len(a)   =", len(a))
print("a[0]     =", a[0])
print("a[3]     =", a[3])
print("a[:2]    =", a[:2])

**和 Max 对照:** 一条 signal patch cord 里流着无数个 sample,你看不到它们的 "编号"。array 是把这些 sample **停下来、排成一排、编上号**,所以可以随便取其中任何一个。

## 2. 三个数字

- `sr`:sample rate,每秒多少个 sample(Max 的 Audio Status 里那个数)
- `freq`:频率,单位 Hz → 决定 **pitch**
- `dur`:时长,单位秒

In [ ]:
sr = 44100
freq = 110
dur = 2.0

## 3. 时间表 `t`

Max 里时间自己在走。Python 里要先把 **每个 sample 发生在第几秒** 写成一张表。

### 先用小数字

假设每秒只有 4 个 sample,一共 2 秒。

- `small_sr * small_dur` = 8 → 一共 8 个 sample
- `np.arange(8)` → 从 0 数到 7,是每个 sample 的编号
- `/ small_sr` → 每个编号除以 4,编号变成秒

In [ ]:
small_sr = 4
small_dur = 2

index = np.arange(int(small_sr * small_dur))   # 每个 sample 的编号:0, 1, 2 ...
small_t = index / small_sr                     # 编号 ÷ 每秒个数 = 第几秒

print("index   =", index)
print("small_t =", small_t)

读一下输出:第 0 个 sample 在 0 秒,第 1 个在 0.25 秒……第 4 个正好在 1 秒(因为每秒 4 个)。

`int(...)` 的作用:`4 * 2` 在这里其实是 `8.0`(带小数点),而 "个数" 必须是整数,所以用 `int` 把它变成 `8`。

### 真实的数字

完全一样的写法,只是 `sr = 44100`。

In [ ]:
# 和上面完全一样:编号 ÷ sr = 每个 sample 的时间(秒)
t = np.arange(int(sr * dur)) / sr

print("number of samples:", len(t))
print("first five:", t[:5])
print("last one:", t[-1])

`t[-1]` 是 "最后一个"。它不是 2.0,而是比 2 秒少一个 sample 的时间 — 因为是从 0 开始数的。

**`t` 里还没有声音,只有时间。**

## 4. 振幅 `x`

现在在每个时间点上填一个振幅。

### 先用小数字

每秒 8 个 sample,频率 1 Hz(每秒 1 个周期),只看 1 秒。

- `freq * t` → 到这个时间点为止,**走了几个周期**
- `2 * np.pi *` → `sin` 只认角度,1 个周期 = 2π。这里只是换单位。
- `np.sin(...)` → sine 在那个位置的高度

In [ ]:
tiny_t = np.arange(8) / 8                 # 1 秒,8 个 sample
cycles = 1 * tiny_t                       # freq × 时间 = 走了几个周期
tiny_x = np.sin(2 * np.pi * cycles)       # 周期数 × 2π = 角度,再取 sine

print("time           :", tiny_t)
print("cycles elapsed :", cycles)
print("amplitude      :", np.round(tiny_x, 2))

读一下 `amplitude` 那一行:0 → 0.71 → 1(最高点)→ 0.71 → 0 → -0.71 → -1(最低点)→ -0.71。正好一个完整的 sine 周期。

(`np.round(..., 2)` 只是为了打印时保留两位小数,好读。)

### 真实的数字

In [ ]:
# freq * t = 走了几个周期;× 2π 换成角度;sin 给出那一刻的振幅
x = np.sin(2 * np.pi * freq * t)

print("number of samples:", len(x))
print("t[100] =", t[100], "   x[100] =", x[100])

### 关于 `t` 和 `x`

- 两个 array 一样长(88200 个数)
- **位置就是编号**:`t[100]` 和 `x[100]` 说的是同一个 sample,一个是它的时间,一个是它的振幅
- `x[100]` 是用 `t[100]` 算出来的,所以顺序自然对得上,不需要另外做标记
- **`x` 就是声音本身。** `t` 只是为了算出 `x`(以及画图的横轴)

## 5. 听(这就是 `dac~`)

`Audio(array, rate=sr)` 会显示一个播放器。

- 第一个位置:要播放的 array
- `rate=sr`:每秒播多少个 sample。它需要这个数才知道用多快的速度播放
- `normalize=False`:不要自动把音量拉到最大

**应该听到:** 一个干净、低沉的 "嗡——",2 秒。

In [ ]:
Audio(0.3 * x, rate=sr, normalize=False)

### 音量

`0.3 * x` 就是 `*~ 0.3`:每个 sample 都乘 0.3。

`x` 原本在 -1 到 1 之间。-1 到 1 是 "满刻度",超过就会 clipping。

**应该听到:** 下面两个音,第一个明显比第二个小声。pitch 完全一样。

In [ ]:
display(Audio(0.05 * x, rate=sr, normalize=False))
display(Audio(0.4 * x, rate=sr, normalize=False))

(一个 cell 里要显示多个播放器时,每个外面包一层 `display(...)`。只有一个时可以不包。)

## 6. 看(这就是 `scope~`)

`plt.plot(横轴, 纵轴)`。只画前 0.03 秒,否则 88200 个点挤在一起什么都看不清。

`n = int(sr * 0.03)` 是 0.03 秒对应多少个 sample;`t[:n]`、`x[:n]` 是各取前 `n` 个。

**先猜再运行:** 110 Hz 的音,0.03 秒里应该有几个周期?

In [ ]:
n = int(sr * 0.03)                # 0.03 秒 = 多少个 sample

plt.figure(figsize=(9, 3))
plt.plot(t[:n], x[:n])            # 横轴:时间,纵轴:振幅,各取前 n 个
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title(f"sine, {freq} Hz")
plt.show()

答案:110 × 0.03 = 3.3 个周期。数一下图里的波峰。

### 放大到能看见 sample

只画前 60 个 sample,并且用圆点标出每一个。**声音其实是这些点,线只是画图时连起来的。**

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(t[:60], x[:60], marker="o", markersize=3)
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title("first 60 samples")
plt.show()

## 7. 包成自己的 object

每次都打 `Audio(0.3 * x, rate=sr, normalize=False)` 很累。在 Max 里你会把常用的一组 object 做成 **abstraction / subpatcher**。Python 里对应的是 **函数**(`def`)。

```python
def 名字(inlet1, inlet2):
    ...做事...
    return outlet
```

- 括号里的是 **inlet**(叫 argument)
- `return` 后面的是 **outlet**
- `gain=0.3` 表示这个 inlet 有默认值,不给就是 0.3
- 缩进的那几行属于这个函数

下面定义三个,之后每一集都会用:

- `play(x)` → `dac~`(里面的 `np.clip` 把超过 ±1 的部分压住,保护耳朵)
- `scope(x)` → `scope~`
- `tone(freq, dur)` → `cycle~` 开一段时间

In [ ]:
def play(x, gain=0.3):
    # gain * x 是音量;np.clip 把超过 ±1 的部分压住
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # 只画开头这么多个 sample
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def tone(freq, dur):
    t = np.arange(int(sr * dur)) / sr          # 这个 t 只在函数里面用,和外面的 t 互不影响
    return np.sin(2 * np.pi * freq * t)

定义函数时 **什么都不会发生**(就像把 abstraction 存了盘但还没放进 patch)。要 "调用" 它才会运行:

**应该听到:** 220 Hz、1 秒的 sine(比刚才高一个八度)。图里 0.03 秒内有 6.6 个周期。

In [ ]:
y = tone(220, 1.0)

scope(y, "tone(220, 1.0)")
play(y)

## 8. 旋律:把 array 接起来

`np.concatenate([a, b, c])` 把几个 array **首尾相接** 成一个更长的 array。

在 Max 里要做旋律得有 `metro`、`counter`、`mtof`…… 在 Python 里,时间就是 array 里的位置,所以 "先后" 就是 "接在后面"。

**应该听到:** 三个音依次响起,A – C# – E(一个上行的大三和弦分解),每个 0.4 秒。

In [ ]:
melody = np.concatenate([tone(220.00, 0.4), tone(277.18, 0.4), tone(329.63, 0.4)])

print("total samples:", len(melody), "=", len(melody) / sr, "seconds")
play(melody)

你可能在音与音交界处听到很轻的 "嗒" 一声。那是因为前一个音没有正好结束在 0 上,波形突然跳了一下。第 5 集的 envelope 会解决它。

## 9. 和弦:把 array 加起来

**接起来 = 先后。加起来 = 同时。**

`a + b` 把两个一样长的 array 逐个 sample 相加。这就是 mixing — 在 Max 里把两根 patch cord 接到同一个 inlet 上,做的就是这件事。

三个 sine 加起来,振幅最大可能到 3,所以音量要调低(`gain=0.1`)。

**应该听到:** 同样三个音,这次同时响,一个 A 大三和弦,2 秒。

In [ ]:
chord = tone(220.00, 2.0) + tone(277.18, 2.0) + tone(329.63, 2.0)

print("max amplitude:", np.max(np.abs(chord)))
scope(chord, "three sines added")
play(chord, gain=0.1)

图里的波形不再是 sine 了 — 三个 sine 叠在一起会形成更复杂的形状。记住这一点,第 3 集会用同样的办法(叠 sine)来设计音色。

## 10. 存成文件

Python 的 output 不只是播放器。`wavfile.write(文件名, sr, array)` 会在这个 notebook 旁边写出一个 wav 文件,可以拖进 Max、Ableton 或任何 DAW。

`.astype(np.float32)` 是把数字存成 32-bit float 格式(wav 的标准格式之一)。

In [ ]:
wavfile.write("ep01_melody.wav", sr, (0.3 * melody).astype(np.float32))
print("saved ep01_melody.wav")

## 11. 一个小实验:`rate` 说谎会怎样

array 只是一串数字,它自己不知道 "每秒该播多少个"。那是 `rate` 告诉播放器的。

如果我们骗它说 sample rate 只有一半:

**先猜再运行:** pitch 会怎样?时长会怎样?

In [ ]:
# sr // 2 = 22050:告诉播放器每秒只播一半的 sample
Audio(0.3 * x, rate=sr // 2, normalize=False)

答案:低一个八度,时长变成 4 秒。同样的数字,播得慢一倍 — 和把磁带放慢、或者 `groove~` 的速度设成 0.5 是同一件事。

## 常见错误

| 现象 | 原因 |
|---|---|
| 改了 `freq` 但声音没变 | 没有重新运行下面算 `x` 的那个 cell |
| `NameError: name 'x' is not defined` | 跳过了上面的 cell,从头按顺序运行一遍 |
| 声音爆掉 / 很刺 | 振幅超过 ±1 了,调低 gain |
| `operands could not be broadcast together` | 两个长度不同的 array 相加了 |

## 练习

**1.** 做一个 440 Hz、0.5 秒的音并播放。

<details><summary>答案</summary>

```python
play(tone(440, 0.5))
```
</details>

**2.** 把第 8 节的旋律改成下行(E – C# – A),每个音 0.25 秒。**应该听到:** 更快的、往下走的三个音。

<details><summary>答案</summary>

```python
play(np.concatenate([tone(329.63, 0.25), tone(277.18, 0.25), tone(220.00, 0.25)]))
```
</details>

**3.** 把 `tone(220, 2.0)` 和 `tone(223, 2.0)` 加起来播放。**先猜:** 会听到什么?

<details><summary>答案</summary>

```python
play(tone(220, 2.0) + tone(223, 2.0), gain=0.15)
```
一个音,音量每秒起伏 3 次(beating)。两个频率差 3 Hz,所以每秒有 3 次互相抵消。
</details>

## 小结

| Max | Python |
|---|---|
| signal(一直在流) | array(一次算完) |
| `cycle~ 110` | `np.sin(2 * np.pi * 110 * t)` |
| `*~ 0.3` | `0.3 * x` |
| 两根线接同一个 inlet | `a + b` |
| `dac~` | `play(x)` |
| `scope~` | `scope(x)` |
| abstraction | `def` 函数 |
| `sfrecord~` | `wavfile.write(...)` |

**下一集:** `phasor~` — 把 pitch 和 timbre 拆开。